# Lab 1 — Data Collection and Pre-Processing
**E-commerce sales dataset: ingest → wrangle → clean → transform → feature-engineer → serialize**

This notebook walks the 12-step Data Engineering road-map end-to-end on a real public sales dataset,
augmented with the fields the assignment requires (`customer_id`, `coupon_code`, `shipping_city`) since the
source file doesn't include them. Every augmentation and every injected data-quality issue is documented
inline so the provenance of every column is traceable.

**Data sources**
1. **Primary**: [ExcelBIAnalytics “1000 Sales Records”](https://excelbianalytics.com/wp/downloads-18-sample-csv-files-data-sets-for-testing-sales/) — `data/1000_sales_records.csv`
2. **Secondary**: a hand-built product catalogue (`data/product_catalog.csv`) mapping each of the 12 `Item Type` values in the primary file to a category, perishability flag, shelf life, and description — used for the Data Dictionary and for feature enrichment.


In [1]:
import json
import random
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("..") / "src") if Path("../src").exists() else "src")
from transaction import Transaction

random.seed(42)
np.random.seed(42)

DATA_DIR = Path("data")
RAW_CSV = DATA_DIR / "1000_sales_records.csv"
CATALOG_CSV = DATA_DIR / "product_catalog.csv"
CLEAN_CSV = DATA_DIR / "cleaned_transactions.csv"
CLEAN_JSON = DATA_DIR / "cleaned_transactions.json"


## Step 1 — Hello, Data!
Load the raw CSV exactly as downloaded and preview it before any modification.

In [2]:
raw_df = pd.read_csv(RAW_CSV)
print(raw_df.shape)
raw_df.head(3)

(1000, 14)


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.20,263.33,3692591.20,2224085.18,1468506.02
1,North America,Canada,Vegetables,Online,M,11/7/2011,185941302,12/8/2011,3018,154.06,90.93,464953.08,274426.74,190526.34
2,Middle East and North Africa,Libya,Baby Food,Offline,C,10/31/2016,246222341,12/9/2016,1517,255.28,159.42,387259.76,241840.14,145419.62


The raw file has 1000 rows and 14 columns (`Region, Country, Item Type, Sales Channel, Order Priority,
Order Date, Order ID, Ship Date, Units Sold, Unit Price, Unit Cost, Total Revenue, Total Cost, Total Profit`).
It does **not** contain `customer_id`, `coupon_code`, or `shipping_city`, which the assignment requires.

Per the assignment's own fallback option ("OR the 500-row synthetic file created in class"), we take the
first 500 rows of the real file and **synthesize the three missing required fields** on top of real
`Country` / `Item Type` / `Unit Price` / `Units Sold` / `Order Date` values, so the dataset stays grounded
in a real public source while still satisfying every required column. We also deliberately inject a handful
of realistic data-quality problems (missing coupons, duplicate rows, bad quantities, messy city text) so
Step 6 (Spot the Grime) has genuine issues to find — this mirrors how the in-class synthetic file was
built and is documented here for full transparency.

In [3]:
df = raw_df.head(500).copy().reset_index(drop=True)

# --- synthesize customer_id: repeating pool of ~120 customers, like a real repeat-purchase base ---
customer_pool = [f"CUST-{i:04d}" for i in range(1, 121)]
df["customer_id"] = np.random.choice(customer_pool, size=len(df))

# --- synthesize shipping_city: one representative city per country in the file ---
country_city_map = {
    country: city
    for country, city in zip(
        df["Country"].unique(),
        [f"{c.split(',')[0][:12]} City" for c in df["Country"].unique()],
    )
}
# Use a small curated lookup for the most common countries, fall back to "<Country> City" otherwise
curated_cities = {
    "Canada": "Toronto", "United States of America": "Chicago", "Mexico": "Mexico City",
    "Libya": "Tripoli", "United Kingdom": "London", "Germany": "Berlin", "France": "Paris",
    "Australia": "Sydney", "India": "Mumbai", "China": "Shanghai", "Japan": "Osaka",
    "Brazil": "Sao Paulo", "South Africa": "Johannesburg", "Nigeria": "Lagos",
}
df["shipping_city"] = df["Country"].map(lambda c: curated_cities.get(c, country_city_map[c]))

# --- synthesize coupon_code: ~55% of orders had no coupon, the rest got one of 5 codes ---
coupon_choices = ["SAVE10", "SAVE20", "WELCOME5", "FREESHIP", "VIP15"]
has_coupon = np.random.rand(len(df)) > 0.55
df["coupon_code"] = [random.choice(coupon_choices) if h else None for h in has_coupon]

# rename/select the fields the assignment asks for, keep useful originals for feature engineering
df = df.rename(columns={
    "Order Date": "date",
    "Item Type": "product",
    "Unit Price": "price",
    "Units Sold": "quantity",
    "Order ID": "order_id",
})
df = df[["order_id", "date", "customer_id", "product", "price", "quantity",
         "coupon_code", "shipping_city", "Country", "Region"]]

# ------------------------------------------------------------------
# Deliberately injected data-quality issues (documented for Step 6/7)
# ------------------------------------------------------------------
rng = np.random.default_rng(7)

# 1) Duplicate rows (5 exact duplicates appended)
dup_rows = df.sample(5, random_state=1)
df = pd.concat([df, dup_rows], ignore_index=True)

# 2) A few missing coupon_code values that are empty strings/whitespace, not just None
messy_idx = rng.choice(df.index, size=6, replace=False)
df.loc[messy_idx, "coupon_code"] = "  "

# 3) A few bad quantity / price values (data entry errors: zero or negative)
bad_idx = rng.choice(df.index, size=4, replace=False)
df.loc[bad_idx, "quantity"] = df.loc[bad_idx, "quantity"] * -1

price_bad_idx = rng.choice(df.index, size=3, replace=False)
df.loc[price_bad_idx, "price"] = 0

# 4) Inconsistent shipping_city casing/whitespace
city_messy_idx = rng.choice(df.index, size=8, replace=False)
df.loc[city_messy_idx, "shipping_city"] = df.loc[city_messy_idx, "shipping_city"].str.upper() + "  "

df = df.reset_index(drop=True)
print(df.shape)
df.head(3)

(505, 10)


,order_id,date,customer_id,product,price,quantity,coupon_code,shipping_city,Country,Region
0,686800706,10/18/2014,CUST-0103,Cosmetics,437.20,8446,SAVE10,Tripoli,Libya,Middle East and North Africa
1,185941302,11/7/2011,CUST-0052,Vegetables,154.06,3018,NaN,Toronto,Canada,North America
2,246222341,10/31/2016,CUST-0093,Baby Food,255.28,-1517,NaN,Tripoli,Libya,Middle East and North Africa


## Step 2 — Pick the Right Container
We use a **`dict`** per record while wrangling (fields are heterogeneous and some are optional/nullable,
so key-based access beats positional access), a **`set`** for uniqueness questions like distinct shipping
cities, and an immutable **`Transaction` dataclass** (below) once a record is considered "final" — a
`namedtuple`/dataclass prevents accidental field drift once cleaning is done. Plain lists/tuples aren't used
as the primary container because we need named, mutable-during-cleaning fields.

## Step 3 — Implement Functions and Data Structure
`Transaction` (in `src/transaction.py`) is a small dataclass with two key methods:
- `.clean()` — normalizes text fields and nulls out invalid numeric values, returns `self` for chaining
- `.total()` — computes discounted revenue for that line, or `None` if data is missing

We populate a list of `Transaction` objects directly from the augmented dataframe.

In [4]:
transactions = [
    Transaction(
        order_id=str(row.order_id),
        date=row.date,
        customer_id=row.customer_id,
        product=row.product,
        price=row.price,
        quantity=row.quantity,
        coupon_code=row.coupon_code,
        shipping_city=row.shipping_city,
    )
    for row in df.itertuples(index=False)
]
print(f"Built {len(transactions)} Transaction objects")
transactions[0]

Built 505 Transaction objects


Transaction(order_id='686800706', date='10/18/2014', customer_id='CUST-0103', product='Cosmetics', price=437.2, quantity=8446, coupon_code='SAVE10', shipping_city='Tripoli', discount_pct=0.0)

## Step 4 — Bulk Loaded
Map the dataframe to a dictionary keyed by `order_id`, which is how a downstream service (e.g. an API
returning one order by id) would typically consume this data.

In [5]:
transactions_by_id = {t.order_id: t.to_dict() for t in transactions}
print(f"{len(transactions_by_id)} unique order_ids indexed")
# peek at one record
next(iter(transactions_by_id.items()))

500 unique order_ids indexed


('686800706',
 {'order_id': '686800706',
  'date': '10/18/2014',
  'customer_id': 'CUST-0103',
  'product': 'Cosmetics',
  'price': 437.2,
  'quantity': 8446,
  'coupon_code': 'SAVE10',
  'discount_pct': 0.0,
  'shipping_city': 'Tripoli',
  'total': 3692591.2})

## Step 5 — Quick Profiling
Basic profiling on the *raw* (pre-clean) augmented data: price range and how many distinct shipping
cities we have (using a `set`, since duplicates/casing noise inflate a naive count).

In [6]:
price_min, price_mean, price_max = df["price"].min(), df["price"].mean(), df["price"].max()
print(f"price -> min={price_min:.2f}, mean={price_mean:.2f}, max={price_max:.2f}")

raw_city_variants = set(df["shipping_city"])
normalized_cities = {c.strip().title() for c in df["shipping_city"]}
print(f"raw distinct shipping_city strings: {len(raw_city_variants)}")
print(f"distinct cities after normalizing case/whitespace: {len(normalized_cities)}")

price -> min=0.00, mean=273.98, max=668.27
raw distinct shipping_city strings: 179
distinct cities after normalizing case/whitespace: 171


## Step 6 — Spot the Grime
At least three real, verifiable dirty-data cases in this dataset:

1. **Duplicate rows** — 5 exact duplicate order rows.
2. **Invalid quantity/price** — a few rows have negative `quantity` or `price == 0` (impossible for a real
   sale).
3. **Inconsistent `shipping_city` text** — mixed case and trailing whitespace (`"TORONTO  "` vs `"Toronto"`)
   that would silently break a naive `groupby`.
4. **Blank-but-not-null `coupon_code`** — whitespace-only strings that `isna()` won't catch.

We verify each of these below before cleaning.

In [7]:
n_dupes = df.duplicated().sum()
n_bad_qty = (df["quantity"] <= 0).sum()
n_bad_price = (df["price"] <= 0).sum()
n_messy_city = df["shipping_city"].ne(df["shipping_city"].str.strip().str.title()).sum()
n_blank_coupon = df["coupon_code"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum()

print(f"duplicate rows:            {n_dupes}")
print(f"quantity <= 0:             {n_bad_qty}")
print(f"price <= 0:                {n_bad_price}")
print(f"messy shipping_city text:  {n_messy_city}")
print(f"blank (whitespace) coupon: {n_blank_coupon}")

duplicate rows:            5
quantity <= 0:             4
price <= 0:                3
messy shipping_city text:  18
blank (whitespace) coupon: 6


## Step 7 — Cleaning Rules
Run `.clean()` on every `Transaction` (normalizes city text, blanks out whitespace-only coupon codes, nulls
invalid price/quantity), drop exact duplicate rows, and drop records that end up invalid
(`is_valid() == False`). Before/after counts are shown explicitly.

In [8]:
before_rows = len(df)
before_dupes = df.duplicated().sum()
before_nulls = df.isna().sum().sum()

# de-duplicate first (dedupe on the business key so exact copies collapse)
df_clean = df.drop_duplicates().reset_index(drop=True)

cleaned_transactions = []
for row in df_clean.itertuples(index=False):
    t = Transaction(
        order_id=str(row.order_id), date=row.date, customer_id=row.customer_id,
        product=row.product, price=row.price, quantity=row.quantity,
        coupon_code=row.coupon_code, shipping_city=row.shipping_city,
    ).clean()
    cleaned_transactions.append(t)

valid_transactions = [t for t in cleaned_transactions if t.is_valid()]

df_clean = pd.DataFrame([t.to_dict() for t in valid_transactions])

after_rows = len(df_clean)
after_dupes = df_clean.duplicated().sum()
after_nulls = df_clean.isna().sum().sum()

print(f"rows:      before={before_rows:4d}  after={after_rows:4d}")
print(f"duplicates: before={before_dupes:4d}  after={after_dupes:4d}")
print(f"null cells: before={before_nulls:4d}  after={after_nulls:4d}")

rows:      before= 505  after= 493
duplicates: before=   5  after=   0
null cells: before= 279  after= 279


## Step 8 — Transformations
Parse `coupon_code` into a numeric `discount_pct` (e.g. `SAVE20` → 20%, `FREESHIP`/`WELCOME5`/`VIP15` use a
fixed lookup, no coupon → 0%), and re-parse `date` into a proper `datetime` column.

In [9]:
discount_lookup = {"SAVE10": 10, "SAVE20": 20, "WELCOME5": 5, "FREESHIP": 0, "VIP15": 15}
df_clean["discount_pct"] = df_clean["coupon_code"].map(discount_lookup).fillna(0)
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")
df_clean["total"] = (df_clean["price"] * df_clean["quantity"] * (1 - df_clean["discount_pct"] / 100)).round(2)
df_clean.head(3)

,order_id,date,customer_id,product,price,quantity,coupon_code,discount_pct,shipping_city,total
0,686800706,2014-10-18,CUST-0103,Cosmetics,437.20,8446,SAVE10,10.0,Tripoli,3323332.08
1,185941302,2011-11-07,CUST-0052,Vegetables,154.06,3018,NaN,0.0,Toronto,464953.08
2,161442649,2010-04-10,CUST-0015,Cereal,205.70,3322,SAVE10,10.0,Osaka,615001.86


## Step 9 — Feature Engineering
Add `days_since_purchase` (relative to the most recent order date in the dataset, so the feature is stable
no matter when the notebook is re-run) and enrich each row with `category` / `is_perishable` from the
secondary product-catalogue file.

In [10]:
catalog = pd.read_csv(CATALOG_CSV)
df_clean = df_clean.merge(catalog[["product", "category", "is_perishable"]], on="product", how="left")

reference_date = df_clean["date"].max()
df_clean["days_since_purchase"] = (reference_date - df_clean["date"]).dt.days
df_clean["is_discounted"] = df_clean["discount_pct"] > 0

df_clean[["product", "category", "is_perishable", "date", "days_since_purchase", "is_discounted"]].head(5)

,product,category,is_perishable,date,days_since_purchase,is_discounted
0,Cosmetics,Personal Care,False,2014-10-18,1012,True
1,Vegetables,Food & Grocery,True,2011-11-07,2088,False
2,Cereal,Food & Grocery,False,2010-04-10,2664,True
3,Fruits,Food & Grocery,True,2011-08-16,2171,True
4,Cereal,Food & Grocery,False,2014-11-24,975,True


## Step 10 — Mini-Aggregation
Total revenue per `shipping_city`, computed two ways to cross-check: a plain dict accumulation and
`pandas.groupby`.

In [11]:
revenue_by_city = {}
for row in df_clean.itertuples(index=False):
    revenue_by_city[row.shipping_city] = revenue_by_city.get(row.shipping_city, 0) + row.total

revenue_by_city_pd = df_clean.groupby("shipping_city")["total"].sum().round(2)

print("dict-based top 5:")
for city, rev in sorted(revenue_by_city.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {city:15s} {rev:>12,.2f}")

print("\ngroupby-based (should match):")
revenue_by_city_pd.sort_values(ascending=False).head(5)

dict-based top 5:
  Papua New Gu City 15,629,197.78
  Costa Rica City 14,465,142.22
  Czech Republ City 13,281,727.03
  Cuba City       12,799,412.05
  Chad City       11,824,090.36

groupby-based (should match):


shipping_city
Papua New Gu City    15629197.78
Costa Rica City      14465142.22
Czech Republ City    13281727.03
Cuba City            12799412.05
Chad City            11824090.36
Name: total, dtype: float64

## Step 11 — Serialization Checkpoint
Persist the cleaned, transformed, feature-engineered dataset in two formats: CSV and JSON.

In [12]:
df_clean.to_csv(CLEAN_CSV, index=False)

records = json.loads(df_clean.to_json(orient="records", date_format="iso"))
with open(CLEAN_JSON, "w") as f:
    json.dump(records, f, indent=2, default=str)

print(f"Saved {len(df_clean)} rows to {CLEAN_CSV} and {CLEAN_JSON}")

Saved 493 rows to data/cleaned_transactions.csv and data/cleaned_transactions.json


## Step 12 — Soft Interview Reflection

Wrapping the record logic in a `Transaction` class with `.clean()` and `.total()` methods meant every one
of the 500+ rows was cleaned and priced through the exact same code path, instead of ad-hoc pandas
one-liners repeated across steps. That made the notebook itself the documentation: reading `.clean()` once
tells you every normalization rule that ever runs, so there's no risk of Step 7 and Step 8 silently
disagreeing about what "clean" means. Functions also made verification cheap — the before/after counts in
Step 7 exist only because `is_valid()` gave a single, reusable definition of a good row. In a real pipeline
this is what lets you unit-test the transformation logic independently of any specific dataframe.

## Data Dictionary
Merged from the **primary** source (raw ExcelBIAnalytics columns + fields synthesized to meet the
assignment's required-field list) and the **secondary** source (`data/product_catalog.csv`).

| Field | Type | Description | Source |
|---|---|---|---|
| `order_id` | string | Unique order identifier. | Primary (raw `Order ID`) |
| `date` | datetime | Date the order was placed. | Primary (raw `Order Date`, parsed in Step 8) |
| `customer_id` | string | Customer identifier. | **Synthetic** — sampled from a 120-customer pool (not present in the raw file) |
| `product` | string | Item type / product name. | Primary (raw `Item Type`) |
| `price` | float | Unit price in USD. | Primary (raw `Unit Price`) |
| `quantity` | int | Units sold on this order line. | Primary (raw `Units Sold`) |
| `coupon_code` | string / null | Promo code applied, if any. | **Synthetic** — randomly assigned from a fixed set of 5 codes, ~45% of rows |
| `discount_pct` | int | Discount percentage implied by `coupon_code`. | **Combination** — derived in Step 8 via a code→percent lookup table |
| `shipping_city` | string | Destination city. | **Synthetic** — one representative city mapped per raw `Country` value |
| `total` | float | `price * quantity * (1 - discount_pct/100)`. | **Combination** — computed in `Transaction.total()` / Step 8 |
| `category` | string | Product category grouping. | Secondary (`product_catalog.csv`, merged on `product` in Step 9) |
| `is_perishable` | bool | Whether the product category is perishable. | Secondary (`product_catalog.csv`, merged on `product` in Step 9) |
| `days_since_purchase` | int | Days between this order's date and the most recent order date in the dataset. | **Derived** — computed in Step 9 |
| `is_discounted` | bool | Whether any coupon discount applied. | **Derived** — computed in Step 9 |
